# Introduction

Pour produire 1kg d'acier, il faut utiliser entre 300 et 600 litres d'eau. L'eau est largement utilisée dans l'industrie pour refroidir les centrales, laver et alimenter les machines... Les usines consomment donc beaucoup d'eau dans leur processus de production.  
Or, les usines ont tendance à s'agglomérer autour de grands sites industriels pour des raisons d'efficacité.  
Nous pouvons donc nous demander quel est l'état des nappes phréatiques autour de ces grands sites industriels. 

Problématique : L'implantation des nouvelles méga-usines (batteries, hydrogène, data centers) se fait-elle de manière aveugle dans des bassins versants déjà condamnés à un épuisement de leurs nappes phréatiques ?

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests as rq
import numpy as np
import time
import os
import contextily as ctx

# Récupération des données

### Récupération des données sur les installations industrielles
On utilise l'API de Georisques pour récupérer les informations sur les usines.

In [ ]:
# On crée le dossier data qui va stocker toutes nos données (gardées pour ne pas les retélécharger à chaque lancement)
dossier_data = "../../../data/"
os.makedirs(dossier_data, exist_ok=True)

#On récupère la liste des installations industrielles à l'échelle nationale sur le site de l'ICPE
url_icpe = "https://mapsref.brgm.fr/wxs/georisques/georisques_dl?&service=wfs&version=2.0.0&request=getfeature&typename=InstallationsClassees&outputformat=CSVTEXT"
chemin_icpe = dossier_data + "icpe.csv"

if not os.path.exists(chemin_icpe):
    fichier_icpe = rq.get(url_icpe, timeout=300)
    fichier_icpe.raise_for_status()   # Erreur si le téléchargement échoue : on n'écrit pas de fichier vide
    with open(chemin_icpe, "wb") as f:
        f.write(fichier_icpe.content)
    print("Fichier téléchargé.")
else:
    print("Le fichier a déjà été téléchargé.")

In [ ]:
# On récupère donc la liste des installations industrielles
installations_indust = pd.read_csv(chemin_icpe, sep=";")
print("On a ", installations_indust.shape[0], "installations industrielles.", end="")
installations_indust = installations_indust.loc[installations_indust["lib_regime"] != "Non ICPE"]
print(" Dont ", installations_indust.shape[0], "installations industrielles ICPE.")

#On veut garder uniquement les installations industrielles pures (pas l'agriculture et l'élevage par ex)
installations_indust = (installations_indust.loc[
    (installations_indust["industrie"] == 1)
    & (installations_indust["bovins"] == 0)
    & (installations_indust["porcs"] == 0)
    & (installations_indust["volailles"] == 0)
    & (installations_indust["eolienne"] == 0)
    ])
print("On ne garde que les ", installations_indust.shape[0], "installations industrielles pures.")

#On classe les départements qui ont le plus d'installations industrielles pour limiter la recherche (car les données sur Hub'eau sont trop volumineuses pour traiter la France entière)
classement_dpt = (installations_indust.groupby("num_dep")
    .size()
    .sort_values(ascending=False)
    )
plt.bar(range(classement_dpt.shape[0]), classement_dpt.values)
plt.title("Nombre d'installations industrielles dans chaque département")
plt.xlabel("Indice des départements (le 1 étant le département avec le plus d'installations)")
plt.ylabel("Nombre d'installations industrielles")

On va donc prendre le département qui contient le plus d'installations industrielles pour notre analyse.  
En effet, l'API Hub'eau nous donne un très grand volume de données et nous devons nous limiter dans notre recherche à quelques départements.  
Avec le graphique ci-dessus, on voit que le département 59 est de loin le plus grand centre industriel français.  
Nous nous concentrerons donc sur le département 59 pour la suite de l'analyse.  

In [ ]:
departement_cible = classement_dpt.index[0]
print("Département analysé : ", departement_cible)

annee_debut, annee_fin = 2015, 2025

Mise en forme des données géographiques avec Geopandas

In [ ]:
installations_indust = installations_indust.loc[installations_indust["num_dep"] == departement_cible]

#On vérifie qu'il n'y a pas d'installations industrielles sans coordonnées géographiques dans la base de données ICPE des installations industrielles du 59
nb_position_less = installations_indust.loc[installations_indust["x"].isnull() | installations_indust["y"].isnull() | installations_indust["code_epsg"].isnull()].shape[0]
print(nb_position_less, "installations industrielles n'ont pas de coordonnées géographiques.")
installations_indust = installations_indust.loc[installations_indust["x"].notnull() & installations_indust["y"].notnull() & installations_indust["code_epsg"].notnull()]

#On utilise les coordonnées des installations pour les localiser avec Geopandas
installations_indust["geometry"] = None #Colonne qui va prendre la valeur position Geopandas
codes_epsg_type_coord = installations_indust["code_epsg"].unique()   #On veut que ça marche pour les différents types de référentiels (x,y)
for epsg in codes_epsg_type_coord:
    filtre = installations_indust["code_epsg"] == epsg
    points = gpd.points_from_xy(
        installations_indust.loc[filtre, "x"], 
        installations_indust.loc[filtre, "y"]
    )
    serie_geometrique = gpd.GeoSeries(points, crs=f"EPSG:{int(epsg)}")
    serie_geometrique = serie_geometrique.to_crs("EPSG:2154") #On ramène tout sous le même format
    installations_indust.loc[filtre, "geometry"] = serie_geometrique.values

#On définit un GeoDataFrame pour les études géométriques :
installations_indust = gpd.GeoDataFrame(installations_indust, geometry="geometry", crs="EPSG:2154")

### Récupération des données sur les nappes phréatiques du 59

L'API Hub'eau ne permet pas d'envoyer toutes les stations d'un coup.  
Encore moins toutes les chroniques d'un coup.  
Donc on sélectionne que la région pour laquelle on a vu qu'il y avait une grande concentration d'industrie : le 59

In [ ]:
# Fonctions de téléchargement Hub'eau, utilisées pour les stations, les chroniques et les prélèvements (partie III)
def requete_hubeau(url, params):
    """Une requête Hub'eau, réessayée en cas de coupure ou de serveur saturé (mais pas en cas de requête refusée)."""
    for essai in range(10):
        try:
            reponse = rq.get(url, params=params, timeout=180)
        except Exception as e:
            print("Erreur requête :", e, "- pas grave, on va réessayer.")
            time.sleep(2)
            continue
        if reponse.status_code in (200, 206) and reponse.headers.get("content-type", "").startswith("application/json"):
            return reponse.json()
        if 400 <= reponse.status_code < 500 and reponse.status_code != 429:   #Requête refusée : réessayer ne servirait à rien
            raise RuntimeError(f"Hub'eau a refusé la requête ({reponse.status_code}) : {reponse.text[:200]}")
        time.sleep(2 * (essai + 1))   #Serveur saturé : on attend un peu plus à chaque fois
    raise RuntimeError("Hub'eau ne répond pas.")

def telecharger_toutes_pages(url, params):
    """Renvoie toutes les lignes d'une requête Hub'eau : l'API découpe les gros résultats en pages, chacune donne l'URL de la suivante dans "next"."""
    lignes = []
    while url is not None:
        page = requete_hubeau(url, params)
        lignes.extend(page["data"])
        url, params = page.get("next"), None   # L'URL "next" contient déjà les paramètres
        time.sleep(0.5)   #On attend un peu pour éviter de se faire rejeter par le serveur
    return lignes


#On récupère les stations de mesure des nappes phréatiques sur le site Hubeau
url_stations = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"
chemin_stations = dossier_data + "stations_nappes.csv"

params = {"code_departement": departement_cible, "size": 5000, "format": "json"}

if not os.path.exists(chemin_stations):
    stations = pd.DataFrame(telecharger_toutes_pages(url_stations, params))
    stations.to_csv(chemin_stations)
    print("On a récupéré ", stations.shape[0], "stations de mesure des nappes phréatiques dans le département ", departement_cible, ".")
else:
    stations = pd.read_csv(chemin_stations)
    print("Le fichier a déjà été téléchargé.")

Mise en forme des données géographiques avec Geopandas

In [ ]:
#On vérifie qu'il n'y a pas d'installations industrielles sans coordonnées géographiques dans la base de données ICPE des installations industrielles du 59
nb_position_less = stations.loc[stations["x"].isnull() | stations["y"].isnull()].shape[0]
print(nb_position_less, "stations n'ont pas de coordonnées géographiques.")
stations = stations.loc[stations["x"].notnull() & stations["y"].notnull()]

#On crée le modèle Geopandas
stations = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations['x'], stations['y']),
    crs="EPSG:4326"
).to_crs(epsg=2154) #On ramène tout sous le même format que les installations industrielles

Pour chaque station identifiée dans les départements contenant le plus d'industrie, on récupère les séries temporelles de niveau des nappes phréatiques entre janvier 2015 et janvier 2026


In [ ]:
chemin_chroniques = dossier_data + f"chroniques_nappes_{departement_cible}.csv"   #Un fichier en cache par département

if not os.path.exists(chemin_chroniques):
    #On ne peut pas récupérer le volume des nappes phréatiques pour toutes les stations en même temps.


    #On veut récupérer les volumes des nappes phréatiques pour chaque station
    url_chroniques = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques"
    chroniques = []

    #Quand on essaye de récupérer toutes les données de toutes les nappes, on a une erreur 414 : la requête URL est trop longue avec tous les code_bss en paramètres.
    #Donc on va diviser les données de 700 stations à récupérer en paquets de 50.

    liste_codes_bss_stations = stations["code_bss"].tolist()
    paquets_intermediaires = []
    for i in range(0, stations.shape[0], 50):
        paquets_intermediaires.append(liste_codes_bss_stations[i:i+50])
    print("Nombre de stations par paquets : ", [len(x) for x in paquets_intermediaires])

    for paquet in paquets_intermediaires:
        params_api = {"code_bss":",".join(paquet), "size":5000, "format":"json", "date_debut_mesure":f"{annee_debut}-01-01", "date_fin_mesure":f"{annee_fin + 1}-01-01"}

        #Le problème est que le volume de données retourné est trop important. => On obtient une erreur 206 pour chaque paquet.
        #Il faut donc récupérer les premières informations renvoyées par l'API et aller chercher les pages suivantes : c'est ce que fait telecharger_toutes_pages.
        chroniques.extend(telecharger_toutes_pages(url_chroniques, params_api))
        print("Paquet téléchargé.")
    print("Téléchargement des donné réussi !")

    chroniques = pd.DataFrame(chroniques)
    chroniques.to_csv(chemin_chroniques)

else:
    chroniques = pd.read_csv(chemin_chroniques)
    print("Le fichier a déjà été téléchargé.")

On nettoie les données des séries temporelles des nappes phréatiques

In [ ]:
chroniques['date_mesure'] = pd.to_datetime(chroniques['date_mesure'])
print("Nombre de chroniques sans niveau d'eau : ", chroniques.loc[chroniques["niveau_nappe_eau"].isnull()].shape[0])
#On supprime les chroniques pour lesquelles le niveau d'eau est NULL
chroniques = chroniques.loc[chroniques["niveau_nappe_eau"].notnull()]

### Test des données

Un exemple de série temporelle pour une station (nappe phréatique)

In [ ]:
# Représentation d'une série temporelle d'une station
station_cible = '00285X0478/PZ10'
serie_station = chroniques.loc[chroniques['code_bss'] == station_cible]
serie_station = serie_station.sort_values(by='date_mesure')

plt.plot(serie_station['date_mesure'], serie_station['niveau_nappe_eau'])

plt.title(f"Évolution du niveau de la nappe - Station {station_cible}", fontsize=14)
plt.xlabel("Date de mesure")
plt.ylabel("Niveau de la nappe d'eau (m)")
plt.grid(True)

### Nettoyage et standardisation des données

Les séries temporelles de Hub'eau donnent des mesures des nappes phréatiques à des dates différentes entre les stations.  
Pour les calculs futurs, on va standardiser ça en faisant une moyenne par station et par mois

In [ ]:
chroniques['date_mois_annee'] = chroniques['date_mesure'].dt.to_period('M')
chroniques

chroniques_niveau_mensuel = (chroniques.groupby(["code_bss", "date_mois_annee"])
              .agg({
                  "niveau_nappe_eau":"mean",
                  "profondeur_nappe":"mean"
              })
              .reset_index()
    )
chroniques_niveau_mensuel["date_mesure"] = chroniques_niveau_mensuel["date_mois_annee"].dt.to_timestamp()
chroniques_niveau_mensuel['date_mesure'] = pd.to_datetime(chroniques_niveau_mensuel['date_mesure'])
chroniques = chroniques_niveau_mensuel.copy()
print("Les données ont été normalisées en moyenne par mois pour chaque station.")

In [ ]:
#Marqueur : cette étape est faite, les notebooks qui en dépendent ne la relanceront pas dans ce noyau
ETAPE_01 = True